# B2-023-generative-models-diffusion — Practice p20 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** stable-diffusion

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
import math
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)

## Solution

**Plan.**
The frozen autoencoder is two module-level functions with no parameters: `encode(x) = x @ E.T` and `decode(z) = z @ E`, with `LATENT_SCALE = 0.25` applied once after encoding and removed once before decoding.
The denoiser is the pinned `LatentDenoiser`: a single-head cross-attention conditioner (query from $(z_t,t/T)$, keys and values from the class token and the learned null token, class score masked to $-\infty$ when dropped) feeding a `19→64→64→2` MLP.
`train_latent_diffusion` encodes the training rows once, then runs exactly 1500 full-batch updates drawing `t`, `eps`, and the conditioning-dropout mask `keep` in that order from one dedicated generator.
The module-level `encode` is wrapped by a recorder for the training call only.
Guided sampling reuses p11's `sample_loop` with a closure that returns `cfg_combine(null-only prediction, conditional prediction, w)`, and the same `x_T`/`noises` are reused for every `w`.

In [ ]:
import copy

DATASET = "latent4d"
T = 50
E = generative_data.encoder_matrix()
LATENT_SCALE = generative_data.LATENT_SCALE


def train_rows():
    return generative_data.train_tensor(DATASET)


def heldout_rows():
    return generative_data.heldout_tensor(DATASET)


def train_labels():
    return generative_data.train_label_tensor(DATASET)


def heldout_labels():
    return generative_data.heldout_label_tensor(DATASET)


def encode(x):
    return x @ E.T


def decode(z):
    return z @ E


def make_schedule(T, beta_1, beta_T):
    if not isinstance(T, int) or isinstance(T, bool) or T < 2:
        raise ValueError("T must be an int >= 2")
    if not (0 < beta_1 <= beta_T < 1):
        raise ValueError("need 0 < beta_1 <= beta_T < 1")
    beta = torch.zeros(T + 1, dtype=torch.float64)
    beta[1:] = torch.linspace(beta_1, beta_T, T, dtype=torch.float64)
    alpha = 1 - beta
    alpha_bar = torch.cumprod(alpha, dim=0)
    beta_tilde = torch.zeros(T + 1, dtype=torch.float64)
    beta_tilde[1:] = (1 - alpha_bar[:-1]) / (1 - alpha_bar[1:]) * beta[1:]
    return {"beta": beta, "alpha": alpha, "alpha_bar": alpha_bar, "beta_tilde": beta_tilde}


def q_sample(x0, t, eps, alpha_bar):
    if x0.shape != eps.shape:
        raise ValueError("x0 and eps must have the same shape")
    if (not isinstance(t, torch.Tensor) or t.dtype != torch.int64 or t.dim() != 1
            or x0.dim() != 2 or t.shape[0] != x0.shape[0]):
        raise ValueError("t must be an int64 tensor of shape (B,)")
    T_max = alpha_bar.shape[0] - 1
    if bool((t < 1).any()) or bool((t > T_max).any()):
        raise ValueError("timesteps must lie in 1..T")
    ab = alpha_bar[t].to(x0.dtype).unsqueeze(1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps


def p_sample(model, x_t, t, z, schedule):
    T_max = schedule["beta"].shape[0] - 1
    if not isinstance(t, int) or isinstance(t, bool) or not 1 <= t <= T_max:
        raise ValueError("t must be an int in 1..T")
    eps_hat = model(x_t, torch.full((x_t.shape[0], 1), t / T_max, dtype=x_t.dtype))
    beta_t = schedule["beta"][t].item()
    alpha_t = schedule["alpha"][t].item()
    alpha_bar_t = schedule["alpha_bar"][t].item()
    mean = (x_t - beta_t / math.sqrt(1 - alpha_bar_t) * eps_hat) / math.sqrt(alpha_t)
    if t == 1:
        return mean
    return mean + math.sqrt(schedule["beta_tilde"][t].item()) * z


def sample_loop(model, x_T, noises, schedule):
    T_max = schedule["beta"].shape[0] - 1
    if tuple(noises.shape) != (T_max, *x_T.shape):
        raise ValueError("noises must have shape (T, *x_T.shape)")
    x = x_T
    for t in range(T_max, 0, -1):
        x = p_sample(model, x, t, noises[t - 1], schedule)
    return x


def cfg_combine(eps_uncond, eps_cond, w):
    if eps_uncond.shape != eps_cond.shape:
        raise ValueError("eps_uncond and eps_cond must have the same shape")
    return eps_uncond + w * (eps_cond - eps_uncond)


class CrossAttentionConditioner(nn.Module):
    def __init__(self, query_in, dim, num_classes):
        super().__init__()
        self.q_proj = nn.Linear(query_in, dim)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.class_tokens = nn.Embedding(num_classes, dim)
        self.null_token = nn.Parameter(torch.zeros(dim))
        self.dim = dim

    def forward(self, query_feats, labels, keep):
        B, dim = query_feats.shape[0], self.dim
        tokens = torch.stack([self.class_tokens(labels), self.null_token.expand(B, dim)], dim=1)
        q = self.q_proj(query_feats).unsqueeze(1)
        k = self.k_proj(tokens)
        v = self.v_proj(tokens)
        scores = q @ k.transpose(1, 2) / math.sqrt(dim)
        drop_class = torch.stack([~keep, torch.zeros_like(keep)], dim=1).unsqueeze(1)
        weights = torch.softmax(scores.masked_fill(drop_class, float("-inf")), dim=-1)
        return (weights @ v).squeeze(1)


class LatentDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.cond = CrossAttentionConditioner(3, 16, 3)
        self.mlp = nn.Sequential(nn.Linear(19, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))

    def forward(self, z_t, t_scaled, labels, keep):
        c = self.cond(torch.cat([z_t, t_scaled], dim=1), labels, keep)
        return self.mlp(torch.cat([z_t, t_scaled, c], dim=1))


def train_latent_diffusion(model, batch, optimizer):
    rows, labels = batch
    z0 = encode(rows) * LATENT_SCALE                 # module-level encode, called once
    B = z0.shape[0]
    noise_generator = torch.Generator().manual_seed(SEED)
    trace = []
    for _ in range(1500):
        t = torch.randint(1, 51, (B,), generator=noise_generator)
        eps = torch.randn(B, 2, generator=noise_generator)
        keep = torch.rand(B, generator=noise_generator) >= 0.2
        optimizer.zero_grad(set_to_none=True)
        z_t = q_sample(z0, t, eps, SCHEDULE["alpha_bar"])
        eps_hat = model(z_t, (t.to(torch.float32) / 50).unsqueeze(1), labels, keep)
        loss = ((eps_hat - eps) ** 2).mean()
        loss.backward()
        optimizer.step()
        trace.append(loss.item())
    return trace


def sample_latents(model, labels, w, x_T, noises):
    n = labels.shape[0]
    all_true = torch.ones(n, dtype=torch.bool)
    all_false = torch.zeros(n, dtype=torch.bool)

    def guided(z, ts):
        return cfg_combine(model(z, ts, labels, all_false), model(z, ts, labels, all_true), w)

    with torch.no_grad():
        return sample_loop(guided, x_T, noises, SCHEDULE)


def split_hash_sets(name):
    hashes = generative_data.ROW_SHA256[name]
    return ({hashes[i] for i in generative_data.TRAIN_IDS[name]},
            {hashes[i] for i in generative_data.HELDOUT_IDS[name]})


TRAIN_HASHES, HELDOUT_HASHES = split_hash_sets(DATASET)
SCHEDULE = make_schedule(50, 1e-3, 0.2)

### Training with the `encode` seam recorded

In [ ]:
torch.manual_seed(SEED)
model = LatentDenoiser()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)

recorded = []
original_encode = encode


def recording_encode(x):
    recorded.append(x.detach().clone())
    return original_encode(x)


encode = recording_encode
try:
    trace = train_latent_diffusion(model, (train_rows(), train_labels()), optimizer)
finally:
    encode = original_encode
seam_restored = encode is original_encode
recorded_rows = torch.cat(recorded)
recorded_hashes = {generative_data.row_sha256(row) for row in recorded_rows}
first50, last50 = sum(trace[:50]) / 50, sum(trace[-50:]) / 50
print(f"first-50 mean {first50:.4f}  last-50 mean {last50:.4f}  ratio {last50 / first50:.3f};  encode calls {len(recorded)}")

### Held-out denoising, guided sampling, and the metrics

In [ ]:
eval_generator = torch.Generator().manual_seed(SEED + 1)
t_eval = torch.randint(1, 51, (48,), generator=eval_generator)
eps_eval = torch.randn(48, 2, generator=eval_generator)
z0_held = encode(heldout_rows()) * LATENT_SCALE
with torch.no_grad():
    z_t_held = q_sample(z0_held, t_eval, eps_eval, SCHEDULE["alpha_bar"])
    ts_held = (t_eval.to(torch.float32) / 50).unsqueeze(1)
    loss_cond = ((model(z_t_held, ts_held, heldout_labels(), torch.ones(48, dtype=torch.bool)) - eps_eval) ** 2).mean().item()
    loss_null = ((model(z_t_held, ts_held, heldout_labels(), torch.zeros(48, dtype=torch.bool)) - eps_eval) ** 2).mean().item()

targets = torch.arange(3).repeat_interleave(100)
sample_generator = torch.Generator().manual_seed(SEED + 2)
x_T = torch.randn(300, 2, generator=sample_generator)
noises = torch.randn(50, 300, 2, generator=sample_generator)
centers = generative_data.class_centers()

dist, hit = {}, {}
for w in (0.0, 1.0, 1.5, 3.0):
    z = sample_latents(model, targets, w, x_T, noises)
    x = decode(z / LATENT_SCALE)
    dist[w] = (x - centers[targets]).norm(dim=1).mean().item()
    nearest = ((x[:, None, :] - centers[None, :, :]) ** 2).sum(dim=2).argmin(dim=1)
    hit[w] = (nearest == targets).float().mean().item()

print(f"held-out loss: conditional {loss_cond:.4f}  null-only {loss_null:.4f}")
for w in dist:
    print(f"w={w:3.1f}: dist {dist[w]:.4f}  hit {hit[w]:.4f}")

### Interpretation

Frozen-seed results (printed by the cell above):

| $w$ | `dist(w)` | `hit(w)` |
|---:|---:|---:|
| 0 | 3.892 | 0.343 |
| 1 | 1.431 | 0.977 |
| 1.5 | 1.276 | 1.000 |
| 3 | 1.329 | 1.000 |

At $w=0$ the sampler uses only the null-only prediction, so it draws from the mixture of all three classes regardless of the target label, and the hit rate is $0.343\approx1/3$, the chance of landing on the requested class by luck.
From $w=1$ to $w=1.5$ almost every sample already lands nearest its target centre, so hit rates saturate near $1$ and can tie or move by a sample or two, whereas the mean distance changes continuously and shows that guidance pulls samples further into their class.
At $w=3$ the extrapolation $\varepsilon_u+3(\varepsilon_c-\varepsilon_u)$ overshoots: samples are pushed past their class's typical region, away from the other classes, so the hit rate stays at $1$ while the mean distance to the target centre rises again (here from $1.276$ at $w=1.5$ to $1.329$).
Guidance only means something if the null-only path was trained (conditioning dropout $0.2$ made one in five training examples use it, which (B)'s null-only held-out loss confirms is a working, if weaker, denoiser); with an untrained null path $\varepsilon_u$ would be garbage and $w\ne1$ would amplify noise rather than class information.

### Answer check

In [ ]:
assert train_rows().shape == (192, 4) and heldout_rows().shape == (48, 4)
assert train_labels().shape == (192,) and heldout_labels().shape == (48,)
assert train_labels().dtype == torch.int64 and heldout_labels().dtype == torch.int64
# (A)
assert len(trace) == 1500 and all(math.isfinite(v) for v in trace)
assert last50 <= 0.75 * first50
# (B)
assert loss_cond <= 0.25 and loss_cond <= loss_null - 0.03
# (C)
assert dist[0.0] >= dist[1.0] + 1.0 and dist[1.0] >= dist[1.5] + 0.05
# (D)
assert hit[1.5] >= hit[1.0] > hit[0.0] and hit[1.0] >= 0.9
# (E)
assert torch.equal(decode(encode(train_rows())), train_rows())
assert torch.equal(decode((encode(train_rows()) * LATENT_SCALE) / LATENT_SCALE), train_rows())
# (F)
assert seam_restored and len(recorded) == 1
assert not (recorded_hashes & HELDOUT_HASHES)
assert TRAIN_HASHES <= recorded_hashes